# Improve a support agent's skill

Our support assistant sends refund and login requests to the wrong team.
We'll improve the short instruction document it reads, using its mistakes to
decide what to change. A **skill** here is a reusable instruction document.

**Start with a skill → propose a correction → test six tickets → inspect what improved.**

This five-cell lesson uses the same tickets and responder as
[Improve a routing prompt](https://sentient-xyz.github.io/meta-evolve-docs/learn/llm-prompt/). You can start here:
every definition is below. The responder and revisions are simple Python
simulations; no model or API key is needed.



## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

<a id="work-through-an-existing-skill-example"></a>

## 2. Start with the skill and six tickets

`SKILL_SEED` maps a document name to its text; no file needs to be created.
The assistant reads that text when answering each ticket. `TICKETS` contains
the expected team for each request.

This simulated assistant already handles invoices and passwords. It handles
refunds and login requests only when those words appear in its instructions.
It does not understand arbitrary text.

In [ ]:
import meta_evolve as meta

# Start with a short instruction document; the name does not create a file.
SKILL_SEED = {
    "routing.md": "Route invoices to billing, passwords to account, otherwise technical.",
}
# Keep these expected teams fixed while we revise the instructions.
TICKETS = (
    ("Please send my invoice", "billing"),
    ("I need a refund", "billing"),
    ("I forgot my password", "account"),
    ("My login is blocked", "account"),
    ("The dashboard freezes", "technical"),
    ("Export is broken", "technical"),
)


# This small simulation stands in for the agent that reads the instructions.
def routing_response(instructions, ticket):
    text = ticket.lower()
    # The simulated agent already handles these two topics.
    if "invoice" in text:
        return "billing"
    if "password" in text:
        return "account"
    # These topics need a matching word in the instruction document.
    if "refund" in instructions and "refund" in text:
        return "billing"
    if "login" in instructions and "login" in text:
        return "account"
    return "technical"  # Send anything else to the technical team.

## 3. Test the answers and record mistakes

`evaluate_skill` asks the assistant to route each ticket, then checks its
answer against the expected team. It records the wrong answers alongside the
score. Meta-Evolve calls these recorded observations **evidence**.

For example, the starting document sends “I need a refund” to `technical`;
the expected answer is `billing`. Both answers go into the feedback.

In [ ]:
def evaluate_skill(skills):
    misses = []
    # Try the current document on every ticket.
    for ticket, expected in TICKETS:
        actual = routing_response(skills["routing.md"], ticket)
        if actual != expected:
            # Save the mistake so the next revision has something to fix.
            misses.append({"ticket": ticket, "expected": expected, "actual": actual})
    # Report both the fraction correct and the mistakes behind that score.
    return meta.EvaluationResult(
        metrics={"score": (len(TICKETS) - len(misses)) / len(TICKETS)},
        evidence=(meta.EvidenceDraft(kind="routing-misses", data={"misses": misses}),),
    )

The tickets, expected answers, and responder stay fixed while the skill changes.

<a id="follow-one-revision"></a>

## 4. Add one instruction from a mistake

`revise_skill` reads the most recent selected feedback and uses its first
miss to write one instruction. For the refund mistake, it adds:
“Route requests like 'I need a refund' to billing.”

`{**skills, ...}` returns a new mapping, preserving the starting document and
any other documents. If feedback is missing, the function reports an error;
if nothing was misrouted, it leaves the skill unchanged.

In [ ]:
def revise_skill(skills, *, context):
    # Read the latest mistakes shared with this revision step.
    feedback = context.evidence.latest("routing-misses")
    if feedback is None:
        raise ValueError("No routing feedback was selected.")
    misses = feedback.data["misses"]
    if not misses:
        return skills  # No mistakes to fix.
    # Turn one missed ticket into an extra instruction.
    miss = misses[0]
    rule = f"Route requests like {miss['ticket']!r} to {miss['expected']}."
    # Build a new version, leaving the previous document intact.
    return {**skills, "routing.md": skills["routing.md"] + "\n" + rule}

<a id="run-the-maintained-example"></a>

## 5. Run, inspect, and use the skill

`RecentAncestors()` makes recent results from the selected version's history
available to the proposer. Recording evidence alone does not supply it as
feedback; this declaration connects the two functions.

`trials=2` allows two proposed revisions after testing the starting document.
Each proposal addresses one recorded miss. Meta-Evolve tests it and keeps the
version with the better score.

In [ ]:
# Test the starting skill, then repeat: revise, test, and keep the best.
skill_result = meta.improve(
    seed=SKILL_SEED,
    proposer=revise_skill,
    evaluator=evaluate_skill,
    trials=2,  # Allow two revisions after checking the starting document.
    context=meta.RecentAncestors(),  # Share recent results with revise_skill.
)

# Follow each version and see which tickets it still gets wrong.
for number, version in enumerate(skill_result.trials()):
    missed = [case["ticket"] for case in version.evidence[0].data["misses"]]
    print(f"Version {number}: {version.metrics['score']:.0%}; misrouted: {missed}")
# Read the selected document and use it for another ticket.
selected_skill = skill_result.best().value
print("Selected routing.md:")
print(selected_skill["routing.md"])
print("Refund please:", routing_response(selected_skill["routing.md"], "Refund please"))
# Output:
# Version 0: 67%; misrouted: ['I need a refund', 'My login is blocked']
# Version 1: 83%; misrouted: ['My login is blocked']
# Version 2: 100%; misrouted: []
# Selected routing.md:
# Route invoices to billing, passwords to account, otherwise technical.
# Route requests like 'I need a refund' to billing.
# Route requests like 'My login is blocked' to account.
# Refund please: billing

The first added instruction fixes refunds; the second fixes login requests.
`selected_skill["routing.md"]` is the document you can pass to your agent.
The original `SKILL_SEED` remains unchanged.

These scores describe the six tickets used to choose revisions, with our
keyword-based simulation. They do not measure a real model or performance on
unseen requests.

## Change and predict

Change `trials=2` to `trials=1` and rerun the last cell. Which mistake
remains, and which instruction was added? You should see **83%**, with login
still misrouted. Set `trials=0` to keep the starting document at **67%**.

To adapt this to your task, replace `TICKETS` with your labeled examples,
`routing_response` with your agent, and the instruction-building part of
`revise_skill` with your revision function. Keep recording actual answers
and checking them independently. [Use your model SDK](https://sentient-xyz.github.io/meta-evolve-docs/guides/providers/) shows
how to connect real calls; include the selected feedback in the revision request.

For these few rules, a handwritten loop is also reasonable. Meta-Evolve keeps
the versions and per-version mistakes, controls which feedback reaches the
proposer, and limits how many revisions it tries. To keep the history after
closing the notebook, follow [Save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/).

<a id="keep-inner-and-outer-evaluation-distinct"></a>
<a id="choose-the-next-capability"></a>
<a id="a-reusable-reading-worksheet"></a>
<a id="complete-supporting-source"></a>
<a id="check-the-learning-journey"></a>

## Go further

- [Choose what to try next](https://sentient-xyz.github.io/meta-evolve-docs/learn/02-change-search/) compares Greedy and
  TreeSearch and follows their recorded parent choices. Search chooses which
  version to revise; feedback chooses which observations reach the proposer.
- [Improve a support agent's playbook](https://sentient-xyz.github.io/meta-evolve-docs/guides/playbook-improvement/) keeps these
  tickets and changes one named rule at a time.
- [Give the proposer feedback](https://sentient-xyz.github.io/meta-evolve-docs/learn/05-use-experience/) explains selected
  history, missing evidence, and what happens when feedback is disabled.
- [The EvoSkill walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/build-patterns/evoskill/) uses three Claude SDK
  roles to turn failed OfficeQA traces into native skill folders. Its
  [standalone notebook](https://sentient-xyz.github.io/meta-evolve-docs/downloads/evoskill.ipynb) and
  [complete source](https://sentient-xyz.github.io/meta-evolve-docs/downloads/evoskill-example.zip) run the live experiment.
- [Compose a method](https://sentient-xyz.github.io/meta-evolve-docs/research/decomposition/#the-worksheet) provides the
  paper-reading worksheet; [improve an agent's harness](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/)
  explores changing how an agent works.

[Previous: Improve a routing prompt](https://sentient-xyz.github.io/meta-evolve-docs/learn/llm-prompt/) ·
[Next: Improve a support agent's playbook](https://sentient-xyz.github.io/meta-evolve-docs/guides/playbook-improvement/)